In [2]:
# 1. Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression


In [3]:
data = pd.read_csv("/content/drive/MyDrive/coding/Ratul bhais paper.csv")

print("data shape:", data.shape)
print(data.head())
print(data.columns)

target_col = "cardiovascular disease"   # change if your column name is different

# drop rows where the target is missing
data = data.dropna(subset=[target_col])

print("after dropping NaN in target, shape:", data.shape)
print(data[target_col].unique())

data shape: (70000, 13)
   Gender  age  height  weight(kg)        BMI  High BP  Low BP  cholesterol  \
0       1   50     168        62.0  21.967120      110      80            1   
1       0   55     156        85.0  34.927679      140      90            3   
2       0   52     165        64.0  23.507805      130      70            3   
3       1   48     169        82.0  28.710479      150     100            1   
4       0   48     156        56.0  23.011177      100      60            1   

   Glucose  smoking  alcohol  active  cardiovascular disease  
0        1        0        0       1                       0  
1        1        0        0       1                       1  
2        1        0        0       0                       1  
3        1        0        0       1                       1  
4        1        0        0       0                       0  
Index(['Gender', 'age', 'height', 'weight(kg)', 'BMI', 'High BP', 'Low BP',
       'cholesterol', 'Glucose', 'smoking', 'al

In [4]:
feature_cols = [
    "Gender", "age", "height", "weight(kg)", "BMI",
    "High BP", "Low BP", "cholesterol", "Glucose",
    "smoking", "alcohol", "active"
]

# Drop rows where any of the feature values is missing
data = data.dropna(subset=feature_cols)
print("After dropping NaN in features:", data.shape)

# Simple automatic outlier trimming using extreme quantiles
num_cols = ["age", "height", "weight(kg)", "BMI",
            "High BP", "Low BP", "cholesterol", "Glucose"]

for col in num_cols:
    low_q, high_q = data[col].quantile([0.001, 0.999])  # keep 0.1%–99.9%
    data = data[(data[col] >= low_q) & (data[col] <= high_q)]

print("After trimming extreme outliers:", data.shape)

X = data[feature_cols]                      # all rows, 12 columns
y = data[target_col].astype(int)           # make sure it is integer 0/1

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels in y:", y.unique())

After dropping NaN in features: (70000, 13)
After trimming extreme outliers: (69394, 13)
X shape: (69394, 12)
y shape: (69394,)
Unique labels in y: [0 1]


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (55515, 12) Test: (13879, 12)


In [6]:
# 5. Define models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),

    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, solver='lbfgs', random_state=42)
    ),

    "Naive Bayes": GaussianNB(),
    "SVM": make_pipeline(
        StandardScaler(),
        SVC(kernel='rbf', probability=True, random_state=42)
    ),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    ),
    "Neural Network": make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(100,),
                      max_iter=1000,
                      random_state=42)
    )
}

# 6. Train and evaluate
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Test accuracy: {acc:.3f}")

Decision Tree Test accuracy: 0.642
Random Forest Test accuracy: 0.711
Logistic Regression Test accuracy: 0.731
Naive Bayes Test accuracy: 0.665
SVM Test accuracy: 0.738
KNN Test accuracy: 0.697
Neural Network Test accuracy: 0.739
